# W10 Practice — Agentic RAG over the EU AI Act

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week10/W10_practice_agentic_rag.ipynb)

**Goal.** The Week 9 program searched once, with the question as the search text, and the model answered from what the search found. This notebook gives the search to the model as a **tool**. The model then decides whether to search, what to search for, whether to search again after reading a result, and when to answer. RAG with these decisions made by the model is called **agentic RAG**. The document, the chunks, the vector store, and the grounding rules are those of Week 9; the setup cell rebuilds them.

| Part | Question |
|---|---|
| 1. The index and the one-search answer | What does the Week 9 program do with a question that needs two passages? |
| 2. The search as a tool | How does a function become a tool that the model can call? |
| 3. The agent | How does the model call the tool, and what does a run look like? |
| 4. A second search | How does the agent follow a reference to another article or annex? |
| 5. The number of searches | How many searches does a question take, and what stops the loop? |
| 6. Memory | How is a follow-up question understood? |

**Cell labels.** The first line of each code cell says what to do with it:

- `# ▶ RUN · example`: read each line, then run.
- `# ▶ RUN · setup`: just run.

It takes about 45 minutes and is not collected. Do Practice 1 (LangChain core patterns) first: `@tool`, `create_agent`, and the message objects come from there. The text of the Act comes from EUR-Lex, without the recitals and the footnotes; only the version in the Official Journal of the EU is legally binding.

## 0. Setup

*Do:* run the four cells; paste your API key into the second. The third cell makes the model object and downloads the text of the Act. The fourth cell rebuilds the index of Week 9 in one go: the split into sections and chunks, the embeddings, the vector store, the retriever with k = 3, `format_passages`, the grounding rules, and `rag_answer`. It takes a few seconds for the embeddings.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "langchain==1.4.4" "langchain-openai==1.7.0" "langgraph==1.2.14" "langchain-text-splitters==1.1.3"

In [ ]:
# ▶ RUN · setup: just run
import os

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # LangChain reads the key from here

In [ ]:
# ▶ RUN · setup: just run
import urllib.request
from langchain.chat_models import init_chat_model

MODEL = "openai:gpt-4.1"
EMBEDDING_MODEL = "text-embedding-3-small"
model = init_chat_model(MODEL)     # one model object for the whole notebook

ACT_PATH = "eu_ai_act.md"
if not os.path.exists(ACT_PATH):   # download the text once
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ralbu85/stml_2026/main/labs/data/eu_ai_act.md", ACT_PATH)
act_text = open(ACT_PATH, encoding="utf-8").read()
print(len(act_text.split()), "words")

In [ ]:
# ▶ RUN · setup: just run
# The index of Week 9 (Practice 2, Parts 2 to 5), in one cell
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.messages import SystemMessage, HumanMessage

header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "chapter"), ("##", "section")])
sections = []
for piece in header_splitter.split_text(act_text):
    if "section" in piece.metadata:                # one Document for each article or annex
        sections.append(piece)
size_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
chunks = size_splitter.split_documents(sections)   # long sections become several chunks

embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = InMemoryVectorStore(embedding_model)
ids = vector_store.add_documents(chunks)           # embeds every chunk once
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print(len(sections), "sections,", len(chunks), "chunks,", len(ids), "in the store")

def format_passages(docs):
    """Write each chunk under its section label, for example "[Article 99: Penalties]"."""
    parts = []
    for doc in docs:
        parts.append("[" + doc.metadata["section"] + "]\n" + doc.page_content)
    return "\n\n".join(parts)

RAG_SYSTEM = """You answer questions about the EU AI Act.
1. Answer only from the passages in the user message. Do not add facts from your own knowledge.
2. After each fact, name its article or annex, for example (Article 99(5)).
3. If the passages do not contain the answer, say so and name what is missing."""

def rag_answer(question, k=3):
    """Week 9: retrieve k chunks for the question, put them into the input, and answer from them."""
    docs = vector_store.as_retriever(search_kwargs={"k": k}).invoke(question)
    messages = [SystemMessage(RAG_SYSTEM), HumanMessage("Passages:\n" + format_passages(docs) + "\n\nQuestion: " + question)]
    return model.invoke(messages).content, docs

## 1. The index and the one-search answer

`rag_answer` is the Week 9 program: one search with the question itself, then one answer from the three chunks that came back. It works when the three chunks hold the answer. Some questions need two passages that no single search returns together, because the second passage is named inside the first one. The Act does this often: Article 6 says which systems are high-risk and points to Annex III for the list; Annex III lists the areas; Article 6(3) names exceptions.

*Do:* run the cell. It asks such a question with the Week 9 program and prints the sections of the three retrieved chunks and the answer. Read the answer against the sections: does it cover both the list and the exception, and does it name a provision for each?

In [ ]:
# ▶ RUN · example: read each line, then run
QUESTION = "A company uses an AI system to rank job applicants. Is it a high-risk AI system, and is there a condition under which it would not be? Name the provisions."

answer_once, docs_once = rag_answer(QUESTION)
for doc in docs_once:
    print("retrieved:", doc.metadata["section"])
print()
print("ANSWER (one search):", answer_once)

**Reading the output.** The three chunks are the three most similar to the whole question. A question with two parts gets chunks for the part that dominates its wording; the provision for the other part may be missing, and then rule 3 applies or the model fills the gap from memory. Keep this answer: Part 4 asks the same question with an agent that can search twice.

## 2. The search as a tool

A **tool** is a function that the model can ask the program to run. In Practice 1 the tool was a weather function; here it is the search of Week 9. `@tool` turns the function into a tool object: the function name becomes the tool name, the type hints become the input schema, and the docstring becomes the description that the model reads. The model never sees the code, only the name, the description, and the schema.

### 2.1 The search tool

*Do:* run the cell. `search_act` runs the retriever for a query and returns the three passages as one text, under their section labels, with `format_passages` of Week 9. The cell prints the tool's name and description, then runs the tool once by hand with `invoke`.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain.tools import tool

@tool
def search_act(query: str) -> str:
    """Search the EU AI Act and return the three most relevant passages, each under its article or annex label."""

    return format_passages(retriever.invoke(query))

print("name:", search_act.name)
print("description:", search_act.description)
print("args:", search_act.args)
print()
print(search_act.invoke("high-risk AI systems in employment")[:900])

**Reading the output.** The description is the docstring, word for word. When the model has this tool, it decides from the description and the question whether to call the tool and with which `query`. The query is written by the model; it need not be the user's question.

## 3. The agent

`create_agent` (Practice 1, Part 6) builds the loop: the model receives the system prompt, the conversation, and the tool descriptions; if its reply requests a tool, the program runs the tool, appends the result as a tool message, and calls the model again; when the reply has no request, the run ends. The system prompt holds the grounding rules of Week 9 plus one rule about the tool and one about the number of searches.

### 3.1 The system prompt and the first run

*Do:* read `AGENT_SYSTEM`, then run the cell. It builds the agent with the search tool and asks the fine question of Week 9. `pretty_print` shows every message of the run: the user message, an AI message with a tool call, the tool message with the passages, and the final AI message.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain.agents import create_agent

AGENT_SYSTEM = """You answer questions about the EU AI Act.
1. Use search_act to find passages. Answer only from the passages you retrieved; do not add facts from your own knowledge.
2. After each fact, name its article or annex, for example (Article 99(5)).
3. If the passages do not contain the answer, say so and name what is missing.
4. Search at most 3 times, then answer from what you have."""

agent = create_agent(model, tools=[search_act], system_prompt=AGENT_SYSTEM)

FINE_QUESTION = "Under the EU AI Act, what is the maximum fine for supplying incorrect or misleading information to national authorities? Name the article."
result = agent.invoke({"messages": [{"role": "user", "content": FINE_QUESTION}]})
for message in result["messages"]:
    message.pretty_print()

**Reading the output.** Four messages. The AI message in the middle has no text, only a tool call: the name `search_act` and the `query` that the model wrote. The tool message is the return value of `search_act`, exactly what `invoke` printed in 2.1. The last AI message is the answer, written after the model read the tool message. Compare the query with the question: the model shortened it to the words that matter for a search. In Week 9 the retriever received the question as it was.

### 3.2 Reading a run

`show_run` prints only the tool calls and the answer of a run, and returns the number of tool calls. The rest of the notebook uses it.

*Do:* run the cell. It defines `show_run` and applies it to the run of 3.1.

In [ ]:
# ▶ RUN · example: read each line, then run
def show_run(result):
    """Print the tool calls of a run and the final answer; return the number of tool calls."""
    calls = 0
    for message in result["messages"]:
        if message.type == "ai" and message.tool_calls:
            for call in message.tool_calls:
                calls = calls + 1
                print("TOOL CALL", calls, ":", call["name"], call["args"])
    print("ANSWER:", result["messages"][-1].content)
    return calls

show_run(result)

## 4. A second search, chosen from the first result

When a passage says "the AI systems referred to in Annex III", the agent needs that annex. It has one tool, the chunk search, so it searches again, with a query that names what the first result pointed to. The retriever is the same; what is new is that the second query is written after the first result has been read. The one-search program of Week 9 could not do this: its only search happened before the model read anything.

### 4.1 A rule for cross-references

*Do:* read the added rule in `AGENT_SYSTEM_2`, then run the cell. It builds the agent again with the new rules and asks the two-part question of Part 1. Read the tool calls: how many searches, and what did the second query ask for? Then compare the answer with the one-search answer of Part 1.

In [ ]:
# ▶ RUN · example: read each line, then run
AGENT_SYSTEM_2 = AGENT_SYSTEM + """
5. When a passage refers to another article or annex that you need, search again with the name of that article or annex as the query."""

agent_2 = create_agent(model, tools=[search_act], system_prompt=AGENT_SYSTEM_2)
result_2 = agent_2.invoke({"messages": [{"role": "user", "content": QUESTION}]})
calls_2 = show_run(result_2)
print()
print("ANSWER (one search, Part 1):", answer_once)

**Reading the output.** The agent chose each query after reading the result of the one before: the question first, then the annex that the first result named. The chunks came from the same retriever as in Week 9; the difference is the second query, which the model wrote. This is what the word *agentic* names: the retriever is a tool, and the model decides the next call from what it has read.

## 5. The number of searches

The number of tool calls is not fixed. It depends on the question: a greeting needs none, a fact in one article needs one, a cross-reference needs two. Two things bound it: rule 4 of the system prompt, which the model follows, and a hard limit of the program, which the model cannot cross.

### 5.1 Four questions

*Do:* run the cell. It sends four questions through the agent and prints the number of tool calls for each. Every run gets the hard limit `HARD_LIMIT`: `recursion_limit` is the maximum number of steps (model calls and tool runs) that one `invoke` may take; the program stops the run when it is reached.

In [ ]:
# ▶ RUN · example: read each line, then run
TEST_QUESTIONS = [
    "Hello! What can you help me with?",
    "What is the maximum fine for using a prohibited AI practice?",
    "Must a chatbot inform people that they are interacting with an AI system, and are there exceptions?",
    "Which obligations apply to deployers of high-risk AI systems, and which annex lists those systems?",
]
HARD_LIMIT = {"recursion_limit": 10}             # at most 10 steps for one run: the program's stop, independent of the prompt

for question in TEST_QUESTIONS:
    print("=" * 100)
    print("QUESTION:", question)
    calls = show_run(agent_2.invoke({"messages": [{"role": "user", "content": question}]}, config=HARD_LIMIT))
    print("tool calls:", calls)

**Reading the output.** The counts differ from question to question, and they can differ between two runs of the same question, because the model chooses. The prompt's "at most 3 times" is a request; `recursion_limit` is the stop that holds even if the model keeps requesting tools.

### 5.2 The hard limit at work

*Do:* run the cell. It asks the fine question with a limit of 2 steps. A run with one search needs three: the model call that requests the search, the tool run, and the model call that writes the answer. The program stops the run after two steps and raises `GraphRecursionError`.

In [ ]:
# ▶ RUN · example: read each line, then run
from langgraph.errors import GraphRecursionError

TIGHT_LIMIT = {"recursion_limit": 2}             # two steps: no step is left for the answer after one search
try:
    show_run(agent_2.invoke({"messages": [{"role": "user", "content": FINE_QUESTION}]}, config=TIGHT_LIMIT))
    print("finished within 2 steps: the model answered without a search")
except GraphRecursionError as error:
    print("stopped by the program:", type(error).__name__)

**Reading the output.** If the model requested a search, the run reached the limit before the answer and the program stopped it: the stop came from `recursion_limit`, not from the prompt. If the model answered without a search, two steps were enough and the run finished.

## 6. Memory

Each `invoke` so far started from an empty conversation: the agent saw only the new question. A **checkpointer** saves the messages of a run under a **thread**, and the next `invoke` with the same `thread_id` continues from them. The model then reads the earlier question, its tool results, and its answer before the new question, so a follow-up such as "and the exceptions?" has something to refer to.

### 6.1 A follow-up question in one thread

*Do:* run the cell. It builds the agent with `InMemorySaver` as the checkpointer, asks a first question in thread `"lesson-1"`, then a follow-up in the same thread, and prints both runs. Then it asks the same follow-up in a new thread, where nothing came before. The last line counts the messages stored in thread `"lesson-1"`.

In [ ]:
# ▶ RUN · example: read each line, then run
from langgraph.checkpoint.memory import InMemorySaver

agent_mem = create_agent(model, tools=[search_act], system_prompt=AGENT_SYSTEM_2, checkpointer=InMemorySaver())
thread_1 = {"configurable": {"thread_id": "lesson-1"}}

FIRST_QUESTION = "Which AI systems in the area of employment are high-risk?"

MY_FOLLOWUP = "Is any of them exempt from the high-risk classification? Name the provision."

print("--- thread lesson-1, first question")
result_first = agent_mem.invoke({"messages": [{"role": "user", "content": FIRST_QUESTION}]}, config=thread_1)
show_run(result_first)
print()
print("--- thread lesson-1, follow-up")
result_follow = agent_mem.invoke({"messages": [{"role": "user", "content": MY_FOLLOWUP}]}, config=thread_1)
show_run(result_follow)
print()
print("--- a new thread, the same follow-up alone")
thread_2 = {"configurable": {"thread_id": "lesson-2"}}
show_run(agent_mem.invoke({"messages": [{"role": "user", "content": MY_FOLLOWUP}]}, config=thread_2))
print()
print("messages stored in thread lesson-1:", len(agent_mem.get_state(thread_1).values["messages"]))

**Reading the output.** In the first thread, the follow-up is answered as a continuation: "them" means the systems of the first answer, and the agent searches for the exception or answers from what it already read. In the new thread, the same words have no referent, so the agent must guess or ask. The stored message count shows the price of memory: every question, tool result, and answer stays in the thread, and the model reads all of it at each call. Chapter 9 is about what to keep in that input, and Week 13 adds the tools that trim and summarize it.